
## 1. Catalyst Optimizer:


- **Catalyst Optimizer** is `Spark SQL's query optimizer`. 
- It improves the `Spark SQL's queries execution plan` to help `Spark SQL's queries run efficiently`.


### Solution 1: If employees_delta is a Delta table:


Use **spark.table()**

In [0]:
df = spark.table("workspace.default.employees_delta")

result = df.filter(df["salary"] > 30000).select("name", "salary")

result.show()


### Solution 2: If you want to read a CSV file:


Use the **actual CSV file path**:

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/data_volume/employees.csv",
    header=True,
    inferSchema=True
)

result = df.filter(df["salary"] > 50000).select("name", "salary")

result.show()


### Types of Catalyst Optimization:


- **1. Rule-Based Optimization (RBO)**
- **2. Cost-Based Optimization (CBO)**


## 1. Rule-Based Optimization (RBO):


- **Rule-Based Optimization** is the `predefined optimization rules`.
- It is used to `improve spark sql query execution without comparing different execution costs`.
- Applies fixed `optimization rules`. 
- `Rule-driven` optimization.

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/data_volume/employees.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
display(df)

In [0]:
from pyspark.sql.functions import col, desc

result = (
    df.filter(col("salary") > 50000)
      .select("name", "salary")
      .orderBy(desc("salary"))
)

display(result)


### First create a temporary view:

In [0]:
df.createOrReplaceTempView("employees")


### Then execute:

In [0]:
%sql

SELECT name, salary
FROM employees
WHERE salary > 50000
ORDER BY salary DESC;


## 2. Cost-Based Optimization (CBO):


- **Cost-Based Optimization** uses `statistics of table`, such as `row counts and data sizes`, 
- It is used to `improve spark sql query execution comparing different execution costs and choose a better plan`.


## Difference between RBO and CBO in Spark SQL (Interview):


| `RBO (Rule-Based Optimizer)` | `CBO (Cost-Based Optimizer)` |
|---|---|
| Uses predefined optimization rules. | Uses statistics and cost estimates. |
| Does not comparing different execution costs. | Compares estimated execution costs. |
| Applies fixed optimization rules. | To estimate the cost of different execution plans and choose a better plan. |
| Rule-driven optimization. | Statistics-driven optimization. |


## 2. Tungsten:


**Tungsten** refers to `Spark SQL's query execution engine` that helps to `reduce memory overhead (RAM)` and `improve CPU efficiency`.

**Example:** 

**Spark can use efficient memory representations and code generation to execute supported operations more efficiently.**


## 3. Explain Plans:


- An **explain plan** is used to `explain how Spark plans to execute a query`. 
- It helps identify `operations such as scans, filters, joins, exchanges, and shuffles`.

In [0]:
df = spark.read.csv("/Volumes/workspace/default/data_volume/employees.csv", header=True, inferSchema=True)

result = df.filter(df.salary > 50000)

result.explain(True)


## 4. Shuffle:


- **Shuffle** is the process of `redistributing data across different partitions or executors` in Spark.
- It is commonly occurs in `wide transformations`.
- Common Operations of shuffle are `groupBy(), distinct(), orderBy(), repartition(), Sort-based joins`.

In [0]:
result = df.groupBy("department").count()

display(result)


## 5. Data skew:


**Data skew** occurs `when data is distributed unevenly across partitions`.

**Example:**

**Suppose we have 1 million employee records distributed across 4 partitions.**

**Balanced data distribution:**

**Partition 1** - 250K records

**Partition 2** - 250K records

**Partition 3** - 250K records

**Partition 4** - 250K records


**Skewed data distribution:**

**Partition 1** - 100K records

**Partition 2** - 100K records

**Partition 3** - 100K records

**Partition 4** - 700K records



### Example:



**Consider an employee dataset with 10,000 records.**

| Department | Records |
|------------|--------:|
| IT         | 8,000   |
| HR         | 1,000   |
| Sales      | 500     |
| Finance    | 500     |
| **Total**  | **10,000** |

Here, IT contains most of the records.

If Spark distributes data by department, the partition processing IT records may receive much more data than the other partitions.


### How to identify Data Skew?

In [0]:
df.groupBy("department").count().display()


### handle Data Skew?

Data Skew is handle by `Salting, Broadcast Join, AQE, Repartitioning, Filtering`.


| Technique | Explanation |
|---|---|
| Salting | Adds `random values to skewed keys` to `distribute data evenly across partitions`. |
| Broadcast Join |` Broadcasts a small table to all worker nodes` to avoid shuffling a large table. |
| AQE | Dynamically handles `skewed partitions` and optimizes `query execution at runtime`. |
| Repartitioning | `Redistributes data across partitions` to `improve parallel processing`. |
| Filtering | `Removes unnecessary records before processing` to `reduce data volume`. |


### Real-time example:


**Suppose an e-commerce company joins orders and customers using customer_id.**

One customer has 1 million orders, while most customers have only 10–20 orders.

This creates data skew because one join key contains significantly more records than others.

Solution: Use `salting` to distribute the heavily repeated customer key across multiple partitions.


## 6. Salting:


- **Salting** is used to `reduce data skew by adding a random or calculated value` to `skewed join keys`. 
- Distributing the `data across multiple partitions`.

In [0]:
from pyspark.sql.functions import *

df = df.withColumn(
    "salt",
    floor(rand() * 5)
)


**This creates salt values from 0 to 4.**

For a skewed join, we can combine the original key with the salt value and replicate matching records on the smaller side of the join.


## 7. Broadcast Join:


A **broadcast join** is used to send a `small dataset to all executors` so that Spark can join it with a `large dataset without shuffling the large dataset`.


### 1. Create employees DataFrame:

In [0]:
employees_data = [
    (1, "Arun Kumar", 55000, 101),
    (2, "Priya Sharma", 62000, 101),
    (3, "Karthik Raj", 48000, 102),
    (4, "Divya Lakshmi", 51000, 103),
    (5, "Suresh Babu", 45000, 104),
    (6, "Meena Iyer", 58000, 105),
    (7, "Rahul Verma", 67000, 101),
    (8, "Anitha Selvam", 47000, 102),
    (9, "Vignesh Kumar", 43000, 106),
    (10, "Deepa Nair", 53000, 103)
]

employees_columns = [
    "employee_id", "name", "salary", "department_id"
]

employees_df = spark.createDataFrame(
    employees_data, employees_columns
)

display(employees_df)


### 2. Create departments DataFrame:

In [0]:
departments_data = [
    (101, "Engineering"),
    (102, "Finance"),
    (103, "Human Resources"),
    (104, "Sales"),
    (105, "Marketing"),
    (106, "Operations")
]

departments_columns = ["department_id", "department_name"]

departments_df = spark.createDataFrame(
    departments_data, departments_columns
)

display(departments_df)


### 3. Perform Broadcast Join:

In [0]:
from pyspark.sql.functions import broadcast

result = employees_df.join(
    broadcast(departments_df),
    "department_id",
    "inner"
)

display(result,truncate=False)


## 8. Repartition vs Coalesce:


### Repartition:


- **repartition()** is used to `increases or decreases a partitions` by `redistributing data across partitions`. 
- Performs a `shuffle`.
- `More expensive`.
- Better for `redistributing data evenly`.	


### Coalesce:


- **coalesce()** is used to `decreases partitions`.
- Usually avoids a `full shuffle`.
- `Less expensive`.
- Better for `reducing the number of partitions`.


### Repartition Example:

In [0]:
df = spark.table("workspace.default.employees_delta")

repartitioned_df = df.repartition(4)

display(repartitioned_df)

print("Repartition completed successfully")


### Coalesce Example:

In [0]:
coalesced_df = df.coalesce(2)

coalesced_df.show()

print("Coalesce completed successfully")


### Compare partition counts without RDD:

In [0]:
from pyspark.sql.functions import spark_partition_id

print("Before repartition:")

df.select(
    spark_partition_id().alias("partition_id")
).distinct().orderBy("partition_id").show()

print("After repartition:")

repartitioned_df.select(
    spark_partition_id().alias("partition_id")
).distinct().orderBy("partition_id").show()


## 9. Caching:


- **Caching** is a process of `store frequently accessed DataFrame data in memory (RAM)` or `on disk`.
- using `df.cache()` in DataFrame API and `CACHE TABLE` in Spark SQL.


### df.cache() vs CACHE TABLE


| df.cache() | CACHE TABLE |
|---|---|
| Used with `DataFrame API`. | Used with `Spark SQL`. |
| Caches a `DataFrame`. | Caches a `table or view`. |
| Example: `df.cache()` | Example: `CACHE TABLE employees` |

In [0]:
df.cache()

df.count()

display(df)

In [0]:
%sql
CACHE TABLE employees;


## 10. Adaptive Query Execution (AQE):


- **Adaptive Query Execution (AQE)** is a `Spark SQL optimization feature`.
- It `optimizes the query execution plan at runtime` using `actual data statistics collected` during execution.


### Important AQE Configurations:


| Configuration | Purpose |
|---|---|
| `spark.sql.adaptive.enabled` | Enables Adaptive Query Execution (AQE). |
| `spark.sql.adaptive.coalescePartitions.enabled` | Enables automatic coalescing of shuffle partitions. |
| `spark.sql.adaptive.skewJoin.enabled` | Enables skew join optimization to handle uneven data distribution. |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | Controls the size threshold for adaptive broadcast joins at runtime. |

In [0]:
spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)


## 11. Small File Problem:


- The **small file problem** occurs `when a large number of small files are generated in storage`, 
- Increasing `metadata overhead and slowing down Spark read operations.`


**Example:**




**Suppose a data pipeline generates 10,000 files of 1 MB each instead of 10 files of 1 GB each.**

Spark must handle many file metadata entries and file-reading operations, which can reduce performance.

How to solve it:
- Use **coalesce()** before writing when appropriate.
- Use **repartition()** to control output partitioning.
- Use **Delta Lake OPTIMIZE** for file compaction.


**Example:**

In [0]:
df.coalesce(5).write \
    .mode("overwrite") \
    .parquet("/data/employees")


**Delta Lake OPTIMIZE:**

In [0]:
OPTIMIZE employees;


**Interview point:** 

I solve the small file problem by **controlling output partitions and using Delta Lake OPTIMIZE** to compact small files into larger files.


### 12. Java and Kryo Serialization:


**Serialization** is the process of `converting objects into a format` that can be `transferred over a network or stored`.


### Java Serialization vs Kryo Serialization


| Java Serialization | Kryo Serialization |
|---|---|
| Default Java serializer in Spark (unless configured otherwise). | Faster alternative in many workloads. |
| Generally slower. | Generally faster. |
| Higher serialization overhead. | Lower serialization overhead. |
| Usually produces larger serialized data. | Generally produces more compact serialized data. |


**Example – Java:**

In [0]:
spark.conf.set(
    "spark.serializer",
    "org.apache.spark.serializer.JavaSerializer"
)


**Example – Kryo:**

In [0]:
spark.conf.set(
    "spark.serializer",
    "org.apache.spark.serializer.KryoSerializer"
)


**Interview point: **

- `Kryo serialization is generally preferred over Java serialization`,
- When `serialization performance and memory efficiency` are important.


## 13. Encoders:


**Encoders** is used to convert `JVM objects into Spark's internal binary representation` and convert `them back when needed`.



**Example:**


**In Scala:**

In [0]:
case class Employee(id: Int, name: String)

val employeeDS = spark.createDataset(
  Seq(Employee(1, "John"))
)


**Interview point:**

Encoders provide `efficient conversion between JVM objects and Spark's internal data format`, especially when working with typed Datasets.


## 14. Partition Pruning:


**Partition pruning** is an `optimization technique` in Spark that `reads only the required partitions` instead of `scanning all partitions`.

**Example:**

SELECT *
FROM sales_partitioned
WHERE year = 2026;

If the table is partitioned by year, Spark can skip partitions belonging to other years.



### Example:




**Suppose employee data is stored in year-wise partitions:**


employees/


    year=2024/

    year=2025/

    year=2026/

In [0]:
df = spark.read.parquet("/data/employees")

result = df.filter("year = 2026")

result.show()


## 15. Partitioning:


- **Partitioning** means dividing `large data into smaller parts`, Spark can process them in `parallel`.
- This can helps spark `to skips unnecessary data` during queries.

**Example:**

**Suppose a sales table is partitioned by year.**

CREATE TABLE sales_partitioned (
    id INT,
    amount DOUBLE,
    year INT
)

**USING DELTA:**

PARTITIONED BY (year);


### Important Partitioning Best Practices:


| Best Practice | Explanation |
|---|---|
| Choose low-cardinality columns | Prefer columns such as year, month or region for partitioning. |
| Avoid high-cardinality columns | Avoid partitioning by employee ID or transaction ID. |
| Target file sizes around 100 MB–1 GB | A practical guideline for many analytical workloads, not a strict Spark rule. |
| Avoid excessive partitions | Too many partitions increase scheduling and metadata overhead. |
| Avoid over-partitioning small data | Small datasets do not need many partitions. |
| Use partition pruning | Filter data using partition columns to reduce unnecessary data reads. |
| Use dynamic partition overwrite | Overwrite only affected partitions when supported and correctly configured. |


**Example:**

In [0]:
df.write \
    .partitionBy("year", "month") \
    .mode("append") \
    .parquet("/data/sales")


**Dynamic partition overwrite:**

In [0]:
spark.conf.set(
    "spark.sql.sources.partitionOverwriteMode",
    "dynamic"
)

df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("/data/sales")


## 16. OPTIMIZE and Compaction in Delta Lake:


**OPTIMIZE** is a `Delta Lake command` used to `improve query performance` by `compacting multiple small files into fewer larger files`.


**Compaction** is used to `combines multiple small files into fewer larger files` to improve `storage and query performance`.


## Why do we need OPTIMIZE?


In **Delta Lake**, frequent `INSERT, UPDATE, DELETE and MERGE operations` can create many small files.


### OPTIMIZE Command:


**Syntax:**

In [0]:
OPTIMIZE sales;


### Compaction for a particular partition:

In [0]:
OPTIMIZE sales
WHERE year = 2026 AND month = 10;


## What is OPTIMIZE in Delta Lake?




**OPTIMIZE** is a `Delta Lake command` used to `improve query performance` by `compacting multiple small files into fewer larger files`.


## What is the difference between OPTIMIZE and ZORDER?


**OPTIMIZE** is a `Delta Lake command` used to `improve query performance` by `compacting multiple small files into fewer larger files`.

**ZORDER** is a `data organization technique` used with `OPTIMIZE` to `colocate related data in the same set of files` based on specified columns.


## 17. Broadcast Hint in DataFrame API vs SQL:


- **Broadcast Join** is a `join optimization technique` in Spark where `a small DataFrame or table is sent to all worker nodes`
- so that it can be joined with `a large DataFrame without shuffling the large dataset` across the network.


## Broadcast Hint in DataFrame API:


In DataFrame API, we use the `broadcast()` function to broadcast a smaller DataFrame during a join.


### Syntax:

In [0]:
from pyspark.sql.functions import broadcast

result = large_df.join(
    broadcast(small_df),
    "department_id",
    "inner"
)


**Example:**

In [0]:
from pyspark.sql.functions import broadcast

result = employees_df.join(
    broadcast(departments_df),
    "department_id",
    "inner"
)

result.show()


## Broadcast Hint in Spark SQL:


In Spark SQL, we use the `BROADCAST` hint to broadcast a smaller table during a join.


### Syntax:

In [0]:
%sql
SELECT /*+ BROADCAST(small_table) */
    *
FROM large_table
JOIN small_table
ON large_table.department_id = small_table.department_id;


### Example:


### First, create temporary views:

In [0]:
employees_df.createOrReplaceTempView("employees")

departments_df.createOrReplaceTempView("departments")


### Execute Spark SQL:

In [0]:
%sql

SELECT /*+ BROADCAST(d) */
    e.employee_id,
    e.name,
    e.salary,
    d.department_name
FROM employees e
JOIN departments d
ON e.department_id = d.department_id;